In [1]:
   import os
   os.chdir("..")
   print(os.getcwd())

/home/s2025ae05941/workspace/heart-mlops


In [2]:
   %%writefile src/predict.py
   """Load the saved model and predict heart-disease risk for new patients.

   Usage:
       python src/predict.py                  # predicts for the built-in sample patient
       python src/predict.py patients.json    # a JSON list of patient records
   """
   import json
   import sys
   from pathlib import Path

   import joblib
   import pandas as pd

   MODEL_PATH = Path("models/model.joblib")

   SAMPLE_PATIENT = {
       "age": 63, "sex": 1, "cp": 1, "trestbps": 145, "chol": 233, "fbs": 1,
       "restecg": 2, "thalach": 150, "exang": 0, "oldpeak": 2.3, "slope": 3,
       "ca": 0, "thal": 6,
   }


   def load_model(path=MODEL_PATH):
       return joblib.load(path)


   def predict(records, model=None):
       """records: list of dicts with the 13 clinical features."""
       model = model or load_model()
       df = pd.DataFrame(records).astype(float)
       proba = model.predict_proba(df)[:, 1]
       results = []
       for p in proba:
           pred = int(p >= 0.5)
           results.append({
               "prediction": pred,
               "label": "heart disease" if pred else "no heart disease",
               "confidence": round(float(max(p, 1 - p)), 3),
               "disease_probability": round(float(p), 3),
           })
       return results


   if __name__ == "__main__":
       if len(sys.argv) > 1:
           patients = json.loads(Path(sys.argv[1]).read_text())
       else:
           patients = [SAMPLE_PATIENT]
       print(json.dumps(predict(patients), indent=2))

Writing src/predict.py


In [3]:
   %run src/predict.py

[
  {
    "prediction": 0,
    "label": "no heart disease",
    "confidence": 0.714,
    "disease_probability": 0.286
  }
]


In [4]:
   import pandas as pd
   from src.predict import predict

   df = pd.read_csv("data/raw/heart.csv")
   sample = df.drop(columns="num").head(10)
   preds = pd.DataFrame(predict(sample.to_dict("records")))
   preds["actual"] = (df["num"].head(10) > 0).astype(int)
   preds

,prediction,label,confidence,disease_probability,actual
0,0,no heart disease,0.714,0.286,0
1,1,heart disease,0.992,0.992,1
2,1,heart disease,0.985,0.985,1
3,0,no heart disease,0.880,0.120,0
4,0,no heart disease,0.938,0.062,0
5,0,no heart disease,0.930,0.070,0
6,1,heart disease,0.765,0.765,1
7,0,no heart disease,0.851,0.149,0
8,1,heart disease,0.916,0.916,1
9,1,heart disease,0.820,0.820,1


The saved pipeline (preprocessing + Logistic Regression) is loaded from models/model.joblib and predicts on raw patient data, so no manual preprocessing is needed. Library versions are pinned in requirements.txt for reproducibility
